# 02 — Modelado · Separar train/test


Split elegido: **80/20 estratificado**. El 20% de test se prioriza para que la estimación final tenga suficientes ictus (~50) y sea estable. La validación cruzada, cuando llegue, subdivide el 80% de train — no crea un tercer bloque.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/raw/stroke_dataset.csv")
df.shape

(4981, 11)

## Features y target

In [3]:


X = df.drop(columns="stroke")
y = df["stroke"]



## Split estratificado 80/20

- `test_size=0.20`: 20% al test.
- `stratify=y`: fuerza la proporción de ictus (~5%) en **ambos** lados. 
- `random_state=42`: fija el sorteo → mismo split en cada corrida (reproducible).

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

## Chequeo de estratificación

In [5]:
resumen = pd.DataFrame(
    {
        "n_filas": [len(y), len(y_train), len(y_test)],
        "n_ictus": [int(y.sum()), int(y_train.sum()), int(y_test.sum())],
        "prop_ictus": [y.mean(), y_train.mean(), y_test.mean()],
    },
    index=["original", "train", "test"],
)
resumen

,n_filas,n_ictus,prop_ictus
original,4981,248,0.049789
train,3984,198,0.049699
test,997,50,0.050150


## Aplicar drops (variables descartadas en el EDA)

Cuatro categóricas se descartaron por planas (`gender`, `Residence_type`) o proxy de edad (`ever_married`, `work_type`).

In [6]:
descartar = ["gender", "Residence_type", "ever_married", "work_type"]

X_train = X_train.drop(columns=descartar)
X_test  = X_test.drop(columns=descartar)

print(X_train.columns.tolist())

['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'smoking_status']


## Codificar smoking_status (one-hot)

Única categórica que sobrevive el drop (`gender`, `Residence_type`, `ever_married`, `work_type` se descartaron por planas o proxy de edad). Es **nominal** (sin orden) → one-hot, no ordinal.

`Unknown` va como categoría de **referencia** (todo-ceros, `drop="Unknown"`): los niveles informativos conservan su peso propio y una fila `Unknown` no aporta señal de tabaquismo — el modelo se apoya en el resto. Decisión conservadora para una herramienta de riesgo desplegable, donde un `Unknown` futuro puede corresponder a un niño pequeño.



In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    categories=[["never smoked", "smokes", "formerly smoked", "Unknown"]],
    drop=["Unknown"],
    sparse_output=False,
    handle_unknown="error",
)

smk_train = encoder.fit_transform(X_train[["smoking_status"]])
smk_test  = encoder.transform(X_test[["smoking_status"]])

## Chequeo — one-hot de smoking_status

Confirmar con la salida real: 3 columnas (4 niveles − 1 baseline) y toda fila `Unknown` sumando 0.

In [8]:
import pandas as pd

cols = encoder.get_feature_names_out(["smoking_status"])
smk_train_df = pd.DataFrame(smk_train, columns=cols, index=X_train.index)

# ¿las filas Unknown quedaron todas en cero?
mask_unknown = X_train["smoking_status"] == "Unknown"
print("columnas:", list(cols))
print("filas Unknown todas en cero:", (smk_train_df[mask_unknown].sum(axis=1) == 0).all())
print(smk_train_df.head())

columnas: ['smoking_status_never smoked', 'smoking_status_smokes', 'smoking_status_formerly smoked']
filas Unknown todas en cero: True
      smoking_status_never smoked  smoking_status_smokes  \
4428                          0.0                    1.0   
1135                          0.0                    0.0   
2417                          0.0                    1.0   
1173                          1.0                    0.0   
3696                          1.0                    0.0   

      smoking_status_formerly smoked  
4428                             0.0  
1135                             0.0  
2417                             0.0  
1173                             0.0  
3696                             0.0  


In [9]:
X_train_final = X_train.drop(columns="smoking_status").join(smk_train_df)

smk_test_df = pd.DataFrame(smk_test, columns=cols, index=X_test.index)
X_test_final = X_test.drop(columns="smoking_status").join(smk_test_df)

print(X_train_final.shape, X_test_final.shape)
print(X_train_final.columns.tolist())

(3984, 8) (997, 8)
['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'smoking_status_never smoked', 'smoking_status_smokes', 'smoking_status_formerly smoked']


## Baseline — Regresión logística (pipeline)



In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

numericas = ["age", "avg_glucose_level", "bmi"]

preprocesado = ColumnTransformer(
    transformers=[("num", StandardScaler(), numericas)],
    remainder="passthrough",
)

modelo_log = Pipeline(steps=[
    ("prep", preprocesado),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

In [11]:
from sklearn.metrics import classification_report, confusion_matrix

modelo_log.fit(X_train_final, y_train)
pred = modelo_log.predict(X_test_final)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

[[701 246]
 [  8  42]]
              precision    recall  f1-score   support

           0      0.989     0.740     0.847       947
           1      0.146     0.840     0.249        50

    accuracy                          0.745       997
   macro avg      0.567     0.790     0.548       997
weighted avg      0.946     0.745     0.817       997



In [12]:
from sklearn.metrics import classification_report
print("TRAIN:\n", classification_report(y_train, modelo_log.predict(X_train_final), digits=3))

TRAIN:
               precision    recall  f1-score   support

           0      0.987     0.730     0.839      3786
           1      0.137     0.818     0.234       198

    accuracy                          0.734      3984
   macro avg      0.562     0.774     0.537      3984
weighted avg      0.945     0.734     0.809      3984

